In [1]:
import sys
import mlflow

print("Version de Python :", sys.version)
print("Environnement Python :", sys.executable)
print("Version de MLflow :", mlflow.__version__)

Version de Python : 3.13.15 (tags/v3.13.15:4061bc4, Aug  5 2026, 13:05:39) [MSC v.1944 64 bit (AMD64)]
Environnement Python : C:\Users\dargu\Documents\Master-SIAD\M2\projet_MLOps\.venv\Scripts\python.exe
Version de MLflow : 3.17.2


### 7.1. Configuration de MLflow

Nous configurons la connexion entre notre notebook et le serveur MLflow local. Nous définissons également une expérience dédiée au projet Bank Marketing, dans laquelle seront enregistrées les différentes expérimentations réalisées.

In [3]:
import mlflow

# Connexion au serveur MLflow local
mlflow.set_tracking_uri("http://127.0.0.1:5000")

# Définition de l'expérience
nom_experience = "Bank_Marketing"

# Création ou récupération de l'expérience
mlflow.set_experiment(nom_experience)

print("Serveur MLflow :", mlflow.get_tracking_uri())
print("Expérience sélectionnée :", nom_experience)

2026/10/30 19:40:42 INFO mlflow.tracking.fluent: Experiment with name 'Bank_Marketing' does not exist. Creating a new experiment.


Serveur MLflow : http://127.0.0.1:5000
Expérience sélectionnée : Bank_Marketing


Nous vérifions que l'expérience a bien été créée sur le serveur MLflow et qu'elle est accessible depuis notre notebook.

In [4]:
# Récupération de l'expérience
experience = mlflow.get_experiment_by_name(nom_experience)

print("Nom :", experience.name)
print("ID :", experience.experiment_id)
print("Statut :", experience.lifecycle_stage)
print("Emplacement des artefacts :", experience.artifact_location)

Nom : Bank_Marketing
ID : 1
Statut : active
Emplacement des artefacts : file:C:/Users/dargu/Documents/Master-SIAD/M2/projet_MLOps/mlartifacts/1


### 7.2. Création d'une première expérimentation

Nous créons une première expérimentation correspondant à l'entraînement de notre régression logistique. Cette exécution servira de référence pour les prochaines expérimentations.

In [5]:
with mlflow.start_run(run_name="Logistic_Regression_Initial"):

    print("Run ID :", mlflow.active_run().info.run_id)
    print("Run Name :", mlflow.active_run().info.run_name)

Run ID : a12819215a0c45b6a1f27a3fc4911b06
Run Name : Logistic_Regression_Initial
🏃 View run Logistic_Regression_Initial at: http://127.0.0.1:5000/#/experiments/1/runs/a12819215a0c45b6a1f27a3fc4911b06
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


### 7.3. Préparation des données

Nous chargeons le jeu de données issu du Feature Engineering et reproduisons la séparation entre les ensembles d'apprentissage et de test.

Nous conservons les mêmes règles de sélection des variables que lors de la première modélisation afin de pouvoir suivre une expérimentation comparable à celle réalisée précédemment.

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

# Chargement des données
chemin_donnees = Path("../data/interim/bank_marketing_feature_engineered.csv")
df = pd.read_csv(chemin_donnees, sep=";")

# Définition de la cible et des variables explicatives
target = "y"

variables_a_exclure = [
    col for col in ["duration", "day_of_week", "pdays"]
    if col in df.columns
]

X = df.drop(columns=[target] + variables_a_exclure)
y = df[target]

print("Dimensions des données :", df.shape)
print("Dimensions de X :", X.shape)
print("Dimensions de y :", y.shape)
print("Variables exclues :", variables_a_exclure)

Dimensions des données : (4119, 21)
Dimensions de X : (4119, 18)
Dimensions de y : (4119,)
Variables exclues : ['duration', 'day_of_week']


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Ensemble d'apprentissage :", X_train.shape)
print("Ensemble de test :", X_test.shape)

Ensemble d'apprentissage : (3295, 18)
Ensemble de test : (824, 18)


### 7.4. Reconstruction du pipeline de modélisation

Nous reconstruisons le pipeline de régression logistique utilisé précédemment. Il comprend le prétraitement des variables numériques et catégorielles ainsi que l'algorithme de classification.

Cette étape permet de reproduire l'entraînement dans le cadre d'une expérimentation suivie par MLflow.

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Identification des types de variables
colonnes_numeriques = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

colonnes_categorielles = [
    col for col in X_train.columns
    if col not in colonnes_numeriques
]

# Prétraitement
preprocesseur = ColumnTransformer(
    transformers=[
        ("numerique", StandardScaler(), colonnes_numeriques),
        ("categorielle", OneHotEncoder(handle_unknown="ignore"),
         colonnes_categorielles)
    ]
)

# Pipeline complet
pipeline_logistique = Pipeline(
    steps=[
        ("pretraitement", preprocesseur),
        ("modele", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ]
)

print("Pipeline reconstruit.")

Pipeline reconstruit.


### 7.5. Tracking de l'entraînement

Nous lançons une nouvelle expérimentation MLflow pour entraîner notre régression logistique.

Nous enregistrons les paramètres du modèle et ceux de la séparation des données. Ces informations permettront de retrouver précisément la configuration utilisée lors de cette exécution.

In [9]:
# Démarrage d'un Run complet
with mlflow.start_run(run_name="Logistic_Regression_Tracked"):

    # Paramètres du modèle
    parametres = {
        "model": "LogisticRegression",
        "max_iter": 1000,
        "random_state": 42,
        "test_size": 0.20,
        "split_random_state": 42,
        "solver": "lbfgs",
        "C": 1.0,
        "n_features": X.shape[1],
        "n_numeric_features": len(colonnes_numeriques),
        "n_categorical_features": len(colonnes_categorielles)
    }

    # Enregistrement des paramètres
    mlflow.log_params(parametres)

    # Entraînement
    pipeline_logistique.fit(X_train, y_train)

    print("Entraînement terminé.")
    print("Run ID :", mlflow.active_run().info.run_id)

Entraînement terminé.
Run ID : e8d394cc17484ffab6b27a0b337b718a
🏃 View run Logistic_Regression_Tracked at: http://127.0.0.1:5000/#/experiments/1/runs/e8d394cc17484ffab6b27a0b337b718a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


### 7.6. Évaluation du modèle

Nous évaluons le modèle à partir de l'ensemble de test. Nous calculons l'accuracy, la précision, le rappel, le F1-score et le ROC-AUC.

Nous étudions également le seuil de classification à partir d'une validation croisée sur l'ensemble d'apprentissage, sans utiliser les données de test pour choisir ce seuil.

### 7.7. Enregistrement des métriques et des artefacts

Nous enregistrons les résultats de l'évaluation dans MLflow. Nous ajoutons également des fichiers complémentaires, notamment la matrice de confusion, la courbe ROC et le rapport de classification.

Ces éléments permettront de retrouver les résultats détaillés de chaque expérimentation directement depuis l'interface MLflow.

### 7.8. Enregistrement du modèle

Nous enregistrons le pipeline entraîné dans MLflow. Il comprend à la fois les transformations appliquées aux variables et le modèle de régression logistique.

Le modèle pourra ainsi être retrouvé et réutilisé à partir de l'expérimentation correspondante.

In [10]:
import tempfile
import json

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

import mlflow.sklearn


with mlflow.start_run(run_name="Logistic_Regression_Tracked"):

    # --------------------------------------------------
    # 1. ENREGISTREMENT DES PARAMÈTRES
    # --------------------------------------------------

    parametres = {
        "model": "LogisticRegression",
        "max_iter": 1000,
        "random_state": 42,
        "test_size": 0.20,
        "split_random_state": 42,
        "solver": "lbfgs",
        "C": 1.0,
        "cv": 5,
        "n_features": X.shape[1],
        "n_numeric_features": len(colonnes_numeriques),
        "n_categorical_features": len(colonnes_categorielles)
    }

    mlflow.log_params(parametres)

    # --------------------------------------------------
    # 2. ENTRAÎNEMENT DU MODÈLE
    # --------------------------------------------------

    pipeline_logistique.fit(X_train, y_train)

    # --------------------------------------------------
    # 3. PRÉDICTIONS SUR LE JEU DE TEST
    # --------------------------------------------------

    y_pred = pipeline_logistique.predict(X_test)

    indice_yes = list(pipeline_logistique.classes_).index("yes")

    y_proba = pipeline_logistique.predict_proba(X_test)[:, indice_yes]

    # --------------------------------------------------
    # 4. CHOIX DU SEUIL PAR VALIDATION CROISÉE
    # --------------------------------------------------

    probabilites_cv = cross_val_predict(
        pipeline_logistique,
        X_train,
        y_train,
        cv=5,
        method="predict_proba"
    )

    indice_yes_cv = list(np.unique(y_train)).index("yes")
    proba_yes_cv = probabilites_cv[:, indice_yes_cv]

    seuils = np.arange(0.10, 0.71, 0.05)

    resultats_seuils = []

    for seuil in seuils:

        predictions_cv = np.where(
            proba_yes_cv >= seuil,
            "yes",
            "no"
        )

        resultats_seuils.append({
            "seuil": seuil,
            "f1": f1_score(
                y_train,
                predictions_cv,
                pos_label="yes",
                zero_division=0
            )
        })

    df_seuils = pd.DataFrame(resultats_seuils)

    meilleur_indice = df_seuils["f1"].idxmax()
    seuil_optimal = float(df_seuils.loc[meilleur_indice, "seuil"])

    y_pred_optimal = np.where(
        y_proba >= seuil_optimal,
        "yes",
        "no"
    )

    # --------------------------------------------------
    # 5. CALCUL DES MÉTRIQUES
    # --------------------------------------------------

    metriques = {
        "accuracy": accuracy_score(y_test, y_pred_optimal),
        "precision": precision_score(
            y_test, y_pred_optimal,
            pos_label="yes",
            zero_division=0
        ),
        "recall": recall_score(
            y_test, y_pred_optimal,
            pos_label="yes",
            zero_division=0
        ),
        "f1_score": f1_score(
            y_test, y_pred_optimal,
            pos_label="yes",
            zero_division=0
        ),
        "roc_auc": roc_auc_score(y_test, y_proba)
    }

    # Enregistrement des métriques
    mlflow.log_metrics(metriques)

    # Enregistrement du seuil retenu
    mlflow.log_param("optimal_threshold", seuil_optimal)

    # --------------------------------------------------
    # 6. ENREGISTREMENT DES ARTEFACTS
    # --------------------------------------------------

    with tempfile.TemporaryDirectory() as dossier_temp:

        dossier_temp = Path(dossier_temp)

        # Matrice de confusion
        fig, ax = plt.subplots(figsize=(6, 5))

        ConfusionMatrixDisplay.from_predictions(
            y_test,
            y_pred_optimal,
            labels=["no", "yes"],
            display_labels=["Non", "Oui"],
            ax=ax
        )

        ax.set_title("Matrice de confusion")
        fig.tight_layout()

        chemin_matrice = dossier_temp / "matrice_confusion.png"
        fig.savefig(chemin_matrice)
        plt.close(fig)

        # Courbe ROC
        fig, ax = plt.subplots(figsize=(6, 5))

        RocCurveDisplay.from_predictions(
            y_test,
            y_proba,
            pos_label="yes",
            ax=ax
        )

        ax.set_title("Courbe ROC - Régression logistique")
        fig.tight_layout()

        chemin_roc = dossier_temp / "courbe_roc.png"
        fig.savefig(chemin_roc)
        plt.close(fig)

        # Rapport de classification
        rapport = classification_report(
            y_test,
            y_pred_optimal,
            zero_division=0
        )

        chemin_rapport = dossier_temp / "rapport_classification.txt"
        chemin_rapport.write_text(rapport, encoding="utf-8")

        # Paramètres et seuil
        chemin_config = dossier_temp / "configuration.json"
        chemin_config.write_text(
            json.dumps(
                {
                    "parametres": parametres,
                    "seuil_optimal": seuil_optimal,
                    "metriques": metriques
                },
                indent=4
            ),
            encoding="utf-8"
        )

        # Envoi des fichiers vers MLflow
        mlflow.log_artifacts(dossier_temp, artifact_path="evaluation")

    # Enregistrement du jeu de données utilisé
    mlflow.log_artifact(
        str(chemin_donnees),
        artifact_path="data"
    )

    # --------------------------------------------------
    # 7. ENREGISTREMENT DU MODÈLE
    # --------------------------------------------------

    mlflow.sklearn.log_model(
        sk_model=pipeline_logistique,
        artifact_path="model"
    )

    print("Tracking terminé.")
    print("Run ID :", mlflow.active_run().info.run_id)
    print("Seuil optimal :", seuil_optimal)
    print("\nMétriques enregistrées :")

    display(pd.DataFrame(
        metriques.items(),
        columns=["Métrique", "Valeur"]
    ).round(3))

2026/10/30 19:49:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Tracking terminé.
Run ID : c00ffca44d284532983d9ecede47b667
Seuil optimal : 0.20000000000000004

Métriques enregistrées :


,Métrique,Valeur
0,accuracy,0.887
1,precision,0.482
2,recall,0.456
3,f1_score,0.469
4,roc_auc,0.775


🏃 View run Logistic_Regression_Tracked at: http://127.0.0.1:5000/#/experiments/1/runs/c00ffca44d284532983d9ecede47b667
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
